In [ ]:
# =========================
# 1. Import Libraries
# =========================
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn import preprocessing
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# =========================
# 2. Upload Dataset (news.csv)
# =========================
from google.colab import files
uploaded = files.upload()

# Load dataset
data = pd.read_csv("news.csv")

# =========================
# 3. Data Preprocessing
# =========================
# Drop unnecessary column
if "Unnamed: 0" in data.columns:
    data = data.drop(["Unnamed: 0"], axis=1)

# Encode labels (REAL = 0, FAKE = 1)
le = preprocessing.LabelEncoder()
data['label'] = le.fit_transform(data['label'])

# =========================
# 4. Variables Setup
# =========================
embedding_dim = 50
max_length = 54
padding_type = 'post'
trunc_type = 'post'
training_size = 3000
test_portion = 0.1

# =========================
# 5. Tokenization
# =========================
title = []
text = []
labels = []

for x in range(training_size):
    title.append(str(data['title'][x]))
    text.append(str(data['text'][x]))
    labels.append(data['label'][x])

tokenizer = Tokenizer()
tokenizer.fit_on_texts(title)

word_index = tokenizer.word_index
vocab_size = len(word_index)

sequences = tokenizer.texts_to_sequences(title)
padded = pad_sequences(sequences, padding=padding_type, truncating=trunc_type)

# =========================
# 6. Train-Test Split
# =========================
split = int(test_portion * training_size)

training_sequences = padded[split:training_size]
test_sequences = padded[0:split]

training_labels = labels[split:training_size]
test_labels = labels[0:split]

training_sequences = np.array(training_sequences)
test_sequences = np.array(test_sequences)

# =========================
# 7. Download GloVe Embeddings
# =========================
!wget http://nlp.stanford.edu/data/glove.6B.zip
!unzip glove.6B.zip

# =========================
# 8. Create Embedding Matrix
# =========================
embedding_index = {}

with open('glove.6B.50d.txt', 'r', encoding='utf-8') as f:
    for line in f:
        values = line.split()
        word = values[0]
        coefs = np.asarray(values[1:], dtype='float32')
        embedding_index[word] = coefs

embedding_matrix = np.zeros((vocab_size + 1, embedding_dim))

for word, i in word_index.items():
    embedding_vector = embedding_index.get(word)
    if embedding_vector is not None:
        embedding_matrix[i] = embedding_vector

# =========================
# 9. Model Architecture
# =========================
model = tf.keras.Sequential([
    tf.keras.layers.Embedding(vocab_size + 1, embedding_dim, input_length=max_length,
                              weights=[embedding_matrix], trainable=False),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Conv1D(64, 5, activation='relu'),
    tf.keras.layers.MaxPooling1D(pool_size=4),
    tf.keras.layers.LSTM(64),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

# =========================
# 10. Train Model
# =========================
history = model.fit(
    training_sequences,
    np.array(training_labels),
    epochs=10,
    validation_data=(test_sequences, np.array(test_labels)),
    verbose=2
)

# =========================
# 11. Prediction Function
# =========================
def predict_news(text_input):
    seq = tokenizer.texts_to_sequences([text_input])
    padded_seq = pad_sequences(seq, maxlen=max_length, padding=padding_type, truncating=trunc_type)

    prediction = model.predict(padded_seq)[0][0]

    if prediction >= 0.5:
        print("REAL NEWS ✅")
    else:
        print("FAKE NEWS ❌")

# =========================
# 12. Test Example
# =========================
predict_news("Breaking: Government announces new economic reforms")

Saving news.csv to news (2).csv
--2026-04-11 08:54:48--  http://nlp.stanford.edu/data/glove.6B.zip
Resolving nlp.stanford.edu (nlp.stanford.edu)... 171.64.67.140
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:80... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://nlp.stanford.edu/data/glove.6B.zip [following]
--2026-04-11 08:54:48--  https://nlp.stanford.edu/data/glove.6B.zip
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip [following]
--2026-04-11 08:54:49--  https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip
Resolving downloads.cs.stanford.edu (downloads.cs.stanford.edu)... 171.64.64.22
Connecting to downloads.cs.stanford.edu (downloads.cs.stanford.edu)|171.64.64.22|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 862182613 (822M) [application/zip]

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │       377,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 377,600 (1.44 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 377,600 (1.44 MB)

Epoch 1/10
85/85 - 4s - 44ms/step - accuracy: 0.6107 - loss: 0.6498 - val_accuracy: 0.6133 - val_loss: 0.6379
Epoch 2/10
85/85 - 1s - 12ms/step - accuracy: 0.6978 - loss: 0.5748 - val_accuracy: 0.7267 - val_loss: 0.5288
Epoch 3/10
85/85 - 1s - 15ms/step - accuracy: 0.7374 - loss: 0.5259 - val_accuracy: 0.7367 - val_loss: 0.5326
Epoch 4/10
85/85 - 2s - 21ms/step - accuracy: 0.7733 - loss: 0.4804 - val_accuracy: 0.7633 - val_loss: 0.4916
Epoch 5/10
85/85 - 2s - 23ms/step - accuracy: 0.8030 - loss: 0.4293 - val_accuracy: 0.7833 - val_loss: 0.4809
Epoch 6/10
85/85 - 2s - 19ms/step - accuracy: 0.8226 - loss: 0.3998 - val_accuracy: 0.7700 - val_loss: 0.4669
Epoch 7/10
85/85 - 1s - 12ms/step - accuracy: 0.8589 - loss: 0.3333 - val_accuracy: 0.7467 - val_loss: 0.5326
Epoch 8/10
85/85 - 1s - 12ms/step - accuracy: 0.8611 - loss: 0.3209 - val_accuracy: 0.7800 - val_loss: 0.5141
Epoch 9/10
85/85 - 1s - 12ms/step - accuracy: 0.8911 - loss: 0.2654 - val_accuracy: 0.7433 - val_loss: 0.6101
Epoch 10/1